# GRPO fine-tuning of the Hangman Transformer — JAX (GPU)

Same algorithm and hyper-parameters as `train_grpo_torch.ipynb`, but end-to-end in JAX:

* the Flax NNX Transformer from `model_jax`, initialised from `weights/aaa_best_weights_nnx.npz`,
* the pure-functional environment `env/hangman_jax.py` (all words live on the GPU),
* one **entire GRPO iteration** (sample words -> rollout -> advantages -> every PPO minibatch update) compiled into a
  single XLA program with `jax.jit` + `jax.lax.scan`, so Python only launches one call per iteration.

Because a GPU steps every game in lockstep, finished games are *frozen* by the environment and masked out of the loss
with the `active` flag, instead of being skipped.

**GRPO in one paragraph.** For each secret word ("prompt") we play a *group* of `G` stochastic games with the
current policy. The reward is `1` for a win and `0` for a loss. Instead of a learned value critic, the baseline is the
group itself: `A_i = (R_i - mean(R_group)) / (std(R_group) + eps)`. Every guess in game `i` is pushed up or down by `A_i`
using the PPO clipped objective, plus a small KL penalty that keeps the policy close to the frozen pre-trained model.

| Step | What happens |
| :--- | :--- |
| 1. Sample | `NUM_PROMPTS` words, each repeated `GROUP_SIZE` times |
| 2. Rollout | Play all games with the sampling policy, store `(obs, mask, action, log_prob, active)` |
| 3. Advantage | Group-normalise the final win/loss reward |
| 4. Update | `PPO_EPOCHS` passes of clipped surrogate + KL over `NUM_MINIBATCHES` minibatches |

### 1. Dataset
Same split (and seed) as `runner_example_notebook.ipynb`: 99% of the words for training, 1% held out for validation.

In [ ]:
import os
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'   # don't grab all VRAM up-front (WSL2 / small GPUs)

import time
import numpy as np
import torch
from torch.utils.data import random_split
from utils.utils import TextDataset

# prepare the dataset (same split as the runner notebook)
dataset = TextDataset('./data/words_250000_train.txt')

train_split_percent = 0.99
train_size = int(train_split_percent * len(dataset))
test_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, test_size], generator=torch.Generator().manual_seed(0))

train_words = [train_dataset[i] for i in range(len(train_dataset))]
val_words = [val_dataset[i] for i in range(len(val_dataset))]
print(f'Training size : {len(train_words)}\nValidation size : {len(val_words)}')

### 2. Hyper-parameters
Identical in the PyTorch and JAX notebooks so the two runs are directly comparable.

In [ ]:
NUM_ITERS       = 50     # GRPO iterations
NUM_PROMPTS     = 32     # distinct words per iteration (scaled up from 16)
GROUP_SIZE      = 8      # games per word  -> 256 games per iteration (scaled up from 128)
PPO_EPOCHS      = 2      # passes over each rollout batch
NUM_MINIBATCHES = 8      # minibatches per pass (384 steps per minibatch in bfloat16)
PACKED_STEPS    = 3072   # fixed active packing window (~46% of 6656 total steps)
LR              = 1e-5
CLIP_EPS        = 0.2    # PPO ratio clipping
KL_BETA         = 0.02   # weight of KL(policy || reference)
EVAL_EVERY      = 10     # greedy evaluation on the full validation set
MAX_STEPS       = 26     # a game can never take more than 26 guesses

### 3. Policy
The policy re-uses the heuristic from the runner notebook: sum the per-position character probabilities over the
(non-pad) positions to get a "letter mass" for `a..z`, mask already-guessed letters, and treat `log(letter_mass)` as
logits. Evaluation takes the `argmax` (exactly the runner's greedy agent); training *samples* from it.

Dropout is set to `0.0` so the rollout policy and the policy being optimised are the same function (PPO ratios start at 1).

In [ ]:
import jax
import jax.numpy as jnp
import optax
from flax import nnx
from model_jax.Models import Transformer

print('JAX devices:', jax.devices())

def load_model():
    model = Transformer(src_vocab=28, d_model=128, max_seq_len=32, N=12, heads=8, dropout=0.0, rngs=nnx.Rngs(0))
    model.load_weights('weights/aaa_best_weights_nnx.npz')
    model.eval()
    bf16_state = jax.tree.map(
        lambda v: v.astype(jnp.bfloat16) if hasattr(v, 'dtype') and jnp.issubdtype(v.dtype, jnp.floating) else v,
        nnx.state(model)
    )
    nnx.update(model, bf16_state)
    return model

policy = load_model()       # trained (bfloat16)
ref_model = load_model()    # frozen reference for the KL term (never passed to the optimizer)
optimizer = nnx.Optimizer(
    policy, optax.chain(optax.clip_by_global_norm(1.0), optax.adamw(LR, weight_decay=0.0)), wrt=nnx.Param
)

def letter_logits(model, x, action_mask):
    """x: (B, 32) observations, action_mask: (B, 26) bool (True = not yet guessed) -> (B, 26) logits."""
    probs = jax.nn.softmax(model(x), axis=-1)             # (B, 32, 28)
    pad_mask = (x != 0)[..., None]                       # (B, 32, 1)
    letter_mass = jnp.sum(probs * pad_mask, axis=1)[:, 1:27]  # (B, 26) for 'a'..'z'
    logits = jnp.log(letter_mass + 1e-12)
    return jnp.where(action_mask, logits, -1e9)          # invalid actions get ~zero probability

def log_prob(logits, actions):
    return jnp.take_along_axis(jax.nn.log_softmax(logits), actions[:, None], axis=-1)[:, 0]

### 4. Rollouts with the JAX environment
`JaxHangmanEnv` pre-encodes the word list on the GPU. `reset_grpo` samples `NUM_PROMPTS` distinct words and tiles each
`GROUP_SIZE` times; `reset_indices` gives a deterministic batch for validation.

`rollout` plays all games for `MAX_STEPS` steps with `jax.lax.scan`. It is a plain traceable function, so it can be
called inside any `jit` (greedy evaluation below, and the training iteration in section 6).

In [ ]:
from env.hangman_jax import JaxHangmanEnv

train_env = JaxHangmanEnv(train_words)
val_env = JaxHangmanEnv(val_words)

def rollout(model, init_states, key, greedy=False):
    """Play every game for MAX_STEPS steps. Returns final states and trajectory leaves of shape (T, B, ...)."""
    def step(carry, _):
        states, key = carry
        key, subkey = jax.random.split(key)
        obs = JaxHangmanEnv.observe(states)
        mask = JaxHangmanEnv.action_mask(states)
        logits = letter_logits(model, obs, mask)
        actions = jnp.argmax(logits, axis=-1) if greedy else jax.random.categorical(subkey, logits)
        record = dict(obs=obs, mask=mask, action=actions, logp=log_prob(logits, actions), active=~states.done)
        next_states, _ = JaxHangmanEnv.step(states, actions)   # finished games stay frozen
        return (next_states, key), record

    (final_states, _), traj = jax.lax.scan(step, (init_states, key), None, length=MAX_STEPS)
    return final_states, traj

@nnx.jit
def greedy_win_rate(model, init_states):
    final_states, _ = rollout(model, init_states, jax.random.PRNGKey(0), greedy=True)
    return final_states.won.mean()

def evaluate(model, env):
    """Greedy win rate (the runner notebook's agent) on every word of `env`."""
    return float(greedy_win_rate(model, env.reset_indices(jnp.arange(env.num_words))))

### 5. GRPO advantage and loss
* **Advantage:** rewards are reshaped to `(NUM_PROMPTS, GROUP_SIZE)` and normalised within each group.
  Groups where every game won (or every game lost) give zero advantage and contribute no policy gradient.
* **Loss:** PPO clipped surrogate `min(r*A, clip(r, 1-eps, 1+eps)*A)` with `r = exp(logp - logp_old)`,
  minus `KL_BETA * KL`, where KL uses the non-negative estimator `exp(ref - logp) - (ref - logp) - 1`.

Shapes stay fixed for XLA, so inactive steps are kept in the batch with weight `0` (masked mean).
The frozen reference model never changes during an iteration, so its log-probs are computed once per rollout
(`ref_logp`) instead of once per minibatch.

In [ ]:
def grpo_advantages(won):
    r = won.astype(jnp.float32).reshape(NUM_PROMPTS, GROUP_SIZE)
    adv = (r - r.mean(axis=1, keepdims=True)) / (r.std(axis=1, keepdims=True) + 1e-6)
    return adv.reshape(-1)                                # (B,)

def grpo_loss(model, batch):
    logp = log_prob(letter_logits(model, batch['obs'], batch['mask']), batch['action'])
    ratio = jnp.exp(logp - batch['logp'])
    adv = batch['adv']
    surrogate = jnp.minimum(ratio * adv, jnp.clip(ratio, 1 - CLIP_EPS, 1 + CLIP_EPS) * adv)

    log_ratio = batch['ref_logp'] - logp
    kl = jnp.exp(log_ratio) - log_ratio - 1

    w = batch['active'].astype(jnp.float32)
    n = jnp.maximum(w.sum(), 1.0)
    return -jnp.sum((surrogate - KL_BETA * kl) * w) / n, jnp.sum(kl * w) / n

### 6. One GRPO iteration as a single compiled function
`jax.jit` needs pure functions of arrays, so the policy + optimizer are split once into a static `graphdef` and an array
`state` (`nnx.split`). `train_iteration` maps `(state, key) -> new state` and contains the whole iteration:

1. sample prompts with `reset_grpo`,
2. roll out all games with the current policy,
3. compute group advantages,
4. pack active steps: sort by `active` and keep a fixed top `PACKED_STEPS = 1536` window, discarding ~54% dead padding rows,
5. precompute frozen reference log-probs once on the packed data,
6. run `PPO_EPOCHS x NUM_MINIBATCHES` gradient steps inside a `jax.lax.scan` over the packed rows.

In [ ]:
train_graphdef, train_state = nnx.split((policy, optimizer))
ref_graphdef, ref_state = nnx.split(ref_model)

@jax.jit
def train_iteration(train_state, ref_state, key):
    k_reset, k_rollout, k_perm = jax.random.split(key, 3)
    policy, _ = nnx.merge(train_graphdef, train_state)
    ref_model = nnx.merge(ref_graphdef, ref_state)

    # 1. sample prompts, each repeated GROUP_SIZE times
    init_states = train_env.reset_grpo(k_reset, NUM_PROMPTS, GROUP_SIZE)

    # 2. rollout with the current policy
    final_states, traj = rollout(policy, init_states, k_rollout)

    # 3. advantages broadcast to every step, flatten (T, B) -> (T * B,)
    T, B = traj['action'].shape
    traj['adv'] = jnp.broadcast_to(grpo_advantages(final_states.won), (T, B))
    data = {k: v.reshape(T * B, *v.shape[2:]) for k, v in traj.items()}

    # 4. Pack active steps: sort by active so all ongoing steps come first, take fixed top PACKED_STEPS
    pack_order = jnp.argsort(~data['active'])[:PACKED_STEPS]
    packed_data = {k: v[pack_order] for k, v in data.items()}

    # 5. Compute reference log-probs once on the packed data
    chunks = {k: v.reshape(NUM_MINIBATCHES, -1, *v.shape[1:]) for k, v in packed_data.items()}
    ref_logp = jax.lax.map(lambda b: log_prob(letter_logits(ref_model, b['obs'], b['mask']), b['action']), chunks)
    packed_data['ref_logp'] = ref_logp.reshape(-1)

    # 6. PPO-style update: every (epoch, minibatch) step inside one lax.scan over PACKED_STEPS
    perms = jax.vmap(lambda k: jax.random.permutation(k, PACKED_STEPS))(jax.random.split(k_perm, PPO_EPOCHS))
    minibatch_idx = perms.reshape(PPO_EPOCHS * NUM_MINIBATCHES, -1)

    def update(train_state, idx):
        policy, optimizer = nnx.merge(train_graphdef, train_state)
        batch = {k: v[idx] for k, v in packed_data.items()}
        (loss, kl), grads = nnx.value_and_grad(grpo_loss, has_aux=True)(policy, batch)
        optimizer.update(policy, grads)
        return nnx.state((policy, optimizer)), (loss, kl)

    train_state, (losses, kls) = jax.lax.scan(update, train_state, minibatch_idx)
    return train_state, dict(reward=final_states.won.mean(), loss=losses[-1], kl=kls[-1])

### 7. Training loop
Evaluate the pre-trained model first, then call `train_iteration` once per iteration. `nnx.update` copies the latest
state back into the `policy` module before evaluating or saving. The first iteration includes XLA compilation time.

In [ ]:
print(f'Baseline greedy validation win rate: {100 * evaluate(policy, val_env):.3f}%')

key = jax.random.PRNGKey(0)
for it in range(1, NUM_ITERS + 1):
    t0 = time.perf_counter()
    key, k_iter = jax.random.split(key)
    train_state, stats = train_iteration(train_state, ref_state, k_iter)
    stats = jax.device_get(stats)                         # waits for the GPU
    print(f'iter {it:3d} | train reward {stats["reward"]:.3f} | loss {stats["loss"]:+.4f} | '
          f'kl {stats["kl"]:.4f} | {time.perf_counter() - t0:.2f} s')

    if it % EVAL_EVERY == 0:
        nnx.update((policy, optimizer), train_state)
        print(f'  -> greedy validation win rate: {100 * evaluate(policy, val_env):.3f}%')

nnx.update((policy, optimizer), train_state)

### 8. Save the fine-tuned weights
Saved in the same flat `.npz` format that `Transformer.load_weights` reads.

In [ ]:
def save_npz_weights(model, path):
    flat = {}
    def walk(tree, prefix):
        for k, v in tree.items():
            name = f'{prefix}.{k}' if prefix else str(k)
            if isinstance(v, dict):
                walk(v, name)
            else:
                arr = np.asarray(v)
                if np.issubdtype(arr.dtype, np.floating) or arr.dtype.name == 'bfloat16':
                    arr = arr.astype(np.float32)
                flat[name] = arr
    walk(nnx.state(model, nnx.Not(nnx.RngState)).to_pure_dict(), '')
    np.savez(path, **flat)

save_npz_weights(policy, 'weights/grpo_jax_weights_nnx.npz')

### Notes
* With only 128 games per iteration the reward curve is noisy; increase `NUM_ITERS` / `NUM_PROMPTS` for a real run.
* The greedy validation win rate of the pre-trained model is the baseline to beat (~66.4%).